# TCP-paper sanity check (standalone)

Reproduces the comparison that justifies dropping TCP from the main grid:

* `garch_hardcoded` — TCP paper's exact GARCH (ω=1e-6, α=0.05, β=0.9, normal, no MLE).
* `garch_roll` — properly fitted rolling GARCH(1,1) Student-t (parametric quantile).
* `fhs` — Filtered Historical Simulation.
* `tcp_paper` — paper-faithful TCP (gradient-boosted quantile + rolling split-conformal).

Headline result: on COVID 2020 individual equities, `garch_roll` and `fhs` both beat `tcp_paper` on Winkler; `garch_hardcoded` undercoverere as expected. Saved to `results/tcp_paper_summary.csv`.

In [ ]:
# sys.path bootstrap so 'from src...' works regardless of CWD
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


In [ ]:
import warnings, glob, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from src.analysis._shared import (
    RESULTS_DIR, CRISIS_WINDOWS, MODEL_CATEGORIES,
    ensure_data, iterate_experiments, walk_forward, save_intervals, make_wrapper,
)
from src.plotting import plot_interval_timeline, compute_cell_metrics, render_cell_grid

FORCE_RERUN = False  # set True to ignore cached CSVs and re-run every cell


In [ ]:
# Reproduces the comparison documented in src/analysis/TCP_Paper/README.md.
# Saved table is also written to results/tcp_paper_summary.csv.
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from src.analysis._shared import ensure_data, iterate_experiments, walk_forward
from src.conformal.garch import GarchHardcoded, RollingGarch
from src.conformal.fhs import FHS
from src.conformal.tcp_paper import TCPPaperConformal
from src.plotting import compute_cell_metrics

CONFIDENCE_LEVEL = 0.90
NUM_LAGS = 5
REFIT_EVERY = 21

data = ensure_data()
TICKERS_2020 = {"MCD", "MGM", "AZO"}
splits = [s for s in iterate_experiments() if s.regime == "2020" and s.ticker.value in TICKERS_2020]

methods = [
    ("garch_hardcoded", lambda lc: GarchHardcoded(label_col_name=lc, confidence_level=CONFIDENCE_LEVEL)),
    ("garch_roll",      lambda lc: RollingGarch(label_col_name=lc, confidence_level=CONFIDENCE_LEVEL,
                                                 garch_window=504, refit_every=REFIT_EVERY)),
    ("fhs",             lambda lc: FHS(label_col_name=lc, confidence_level=CONFIDENCE_LEVEL)),
    ("tcp_paper",       lambda lc: TCPPaperConformal(label_col_name=lc, num_lags=NUM_LAGS,
                                                      confidence_level=CONFIDENCE_LEVEL,
                                                      refit_every=REFIT_EVERY)),
]

rows = []
for s in splits:
    train = s.slice_train(data); calib = s.slice_calib(data); full = s.slice_full(data)
    for name, factory in methods:
        m = factory(s.label_col)
        if name == "fhs":
            m.fit_and_calibrate(pd.concat([train, calib]))
        else:
            m.fit_and_calibrate(train, calib)
        intervals = walk_forward(m, full, s.test_start, s.test_end)
        cell = compute_cell_metrics(intervals)
        rows.append({"method": name, "ticker": s.ticker.value,
                     "cov": cell["cov"], "winkler": cell["winkler"], "width": cell["width"]})

per_cell = pd.DataFrame(rows)
display(per_cell.round(3))

summary = (per_cell.groupby("method")[["cov", "winkler", "width"]]
           .mean().sort_values("winkler").round(3))

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
summary.to_csv(RESULTS_DIR / "tcp_paper_summary.csv")
print(f"\nSaved summary → {RESULTS_DIR/'tcp_paper_summary.csv'}")
print("\nAverage across MCD, MGM, AZO  (lower Winkler = better)")
display(summary)
